In [1]:
!pip install codecarbon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.8/179.8 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 6.5 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import confusion_matrix, classification_report
import warnings
import os

import seaborn as sns
sns.set()

# disabling the display of warnings
warnings.filterwarnings('ignore')
from codecarbon import EmissionsTracker
from codecarbon import track_emissions
import scipy.stats as stats

sns.set()

In [3]:

original_data = pd.read_csv('heart_2020.csv')

In [4]:
original_data.head()

,HeartDisease,BMI,Smoking,AlcoholDrinking,Stroke,PhysicalHealth,MentalHealth,DiffWalking,Sex,AgeCategory,Race,Diabetic,PhysicalActivity,GenHealth,SleepTime,Asthma,KidneyDisease,SkinCancer
0,No,16.60,Yes,No,No,3.0,30.0,No,Female,55-59,White,Yes,Yes,Very good,5.0,Yes,No,Yes
1,No,20.34,No,No,Yes,0.0,0.0,No,Female,80 or older,White,No,Yes,Very good,7.0,No,No,No
2,No,26.58,Yes,No,No,20.0,30.0,No,Male,65-69,White,Yes,Yes,Fair,8.0,Yes,No,No
3,No,24.21,No,No,No,0.0,0.0,No,Female,75-79,White,No,No,Good,6.0,No,No,Yes
4,No,23.71,No,No,No,28.0,0.0,Yes,Female,40-44,White,No,Yes,Very good,8.0,No,No,No


In [5]:
original_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 319795 entries, 0 to 319794
Data columns (total 18 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   HeartDisease      319795 non-null  object 
 1   BMI               319795 non-null  float64
 2   Smoking           319795 non-null  object 
 3   AlcoholDrinking   319795 non-null  object 
 4   Stroke            319795 non-null  object 
 5   PhysicalHealth    319795 non-null  float64
 6   MentalHealth      319795 non-null  float64
 7   DiffWalking       319795 non-null  object 
 8   Sex               319795 non-null  object 
 9   AgeCategory       319795 non-null  object 
 10  Race              319795 non-null  object 
 11  Diabetic          319795 non-null  object 
 12  PhysicalActivity  319795 non-null  object 
 13  GenHealth         319795 non-null  object 
 14  SleepTime         319795 non-null  float64
 15  Asthma            319795 non-null  object 
 16  KidneyDisease     31

In [6]:
original_data.isnull().sum()

HeartDisease        0
BMI                 0
Smoking             0
AlcoholDrinking     0
Stroke              0
PhysicalHealth      0
MentalHealth        0
DiffWalking         0
Sex                 0
AgeCategory         0
Race                0
Diabetic            0
PhysicalActivity    0
GenHealth           0
SleepTime           0
Asthma              0
KidneyDisease       0
SkinCancer          0
dtype: int64

In [7]:
from sklearn.preprocessing import LabelEncoder

categorical_columns = original_data.select_dtypes(include=['object']).columns

label_encoder = LabelEncoder()

for column in categorical_columns:
    original_data[column] = label_encoder.fit_transform(original_data[column])

original_data = original_data.astype(float)
original_data.head()



,HeartDisease,BMI,Smoking,AlcoholDrinking,Stroke,PhysicalHealth,MentalHealth,DiffWalking,Sex,AgeCategory,Race,Diabetic,PhysicalActivity,GenHealth,SleepTime,Asthma,KidneyDisease,SkinCancer
0,0.0,16.60,1.0,0.0,0.0,3.0,30.0,0.0,0.0,7.0,5.0,2.0,1.0,4.0,5.0,1.0,0.0,1.0
1,0.0,20.34,0.0,0.0,1.0,0.0,0.0,0.0,0.0,12.0,5.0,0.0,1.0,4.0,7.0,0.0,0.0,0.0
2,0.0,26.58,1.0,0.0,0.0,20.0,30.0,0.0,1.0,9.0,5.0,2.0,1.0,1.0,8.0,1.0,0.0,0.0
3,0.0,24.21,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.0,5.0,0.0,0.0,2.0,6.0,0.0,0.0,1.0
4,0.0,23.71,0.0,0.0,0.0,28.0,0.0,1.0,0.0,4.0,5.0,0.0,1.0,4.0,8.0,0.0,0.0,0.0


In [8]:
 # define models

clf_rdf = RandomForestClassifier()
clf_dtc = DecisionTreeClassifier()
clf_xgb = xgb.XGBClassifier()
clf_lr = LogisticRegression()
clf_knn = KNeighborsClassifier(n_neighbors=3)
clf_svc = SVC()
clf_gnb = GaussianNB()

In [9]:
def model_fit(model_name, model, num_features, reduction_percentage, X_train, y_train, run):
    project_name = f"{model_name}_{num_features}_features_reduced_by_{reduction_percentage}_percent{run}"
    print(model_name)

    tracker = EmissionsTracker(project_name=project_name)
    tracker.start()
    model.fit(X_train, y_train)
    tracker.stop()

In [10]:
# csv adaption
def csv_adaption(model_name, model, num_features, reduction_percentage, X_train, y_train, X, run):
    project_name = f"{model_name}_{num_features}_features_reduced_by_{reduction_percentage}_percent{run}"
    columns_names = ', '.join(X.columns)


    if os.path.exists('emissions_data.csv'):
      dataf = pd.read_csv('emissions_data.csv')
      new_row = {'project_name': project_name,'num_features': num_features,'reduction_percentage': reduction_percentage, 'feature_names': columns_names}
      idx = len(dataf)  # Bestimme den Index für die neue Zeile
      dataf.loc[idx] = new_row
      #dataf.append(new_row, ignore_index=True)
      dataf.to_csv('emissions_data.csv', index=False)

    else:
      data ={
        'project_name': [project_name],
        'num_features': [num_features],
        'reduction_percentage': [reduction_percentage],
        'feature_names': [columns_names]
        }
      dataf = pd.DataFrame(data)
      dataf.to_csv('emissions_data.csv', index=False)


In [11]:
# row reduction
def reduce_dataset(dataframe, reduction_percentage):
    num_samples = int(len(dataframe) * (reduction_percentage / 100))
    reduced_dataframe = dataframe.sample(frac=(1 - (num_samples / len(dataframe))))

    return reduced_dataframe

In [13]:
def row_main_variation(data, model_name, model, num_features, run):

    for i in reduction_array:

        print()
        print("Dataset reduced by [%]:", i)

        df_reduced_samples = reduce_dataset(data, i)
        num_rows = df_reduced_samples.shape[0]
        print("Number of rows:", num_rows)

        # feature to be predicted
        y = df_reduced_samples["HeartDisease"]
        #print(y)
        # all other features are used as predictors
        X = df_reduced_samples.drop(labels = ["HeartDisease"], axis = 1)

        # split data
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=110)

        # scale data
        scaler = StandardScaler()
        scaler.fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

        # train models
        model_fit(model_name, model, num_features, i, X_train, y_train, run)
        csv_adaption(model_name, model, num_features, i, X_train, y_train, X, run)

In [15]:
# feature reduction
reduction_array = [0,20,40,60,80]
#clf_names = ['RandomForest', 'DecisionTree', 'XGBoost', 'LogisticRegression', 'KNeighborsClassifier', 'GaussianNB']
#clf_names = ['RandomForest']
#clf_names = ['DecisionTree']
#clf_names = ['XGBoost']
clf_names = ['LogisticRegression']
#clf_names = ['KNeighborsClassifier']
#clf_names = ['GaussianNB']

#clf_list = [clf_rdf, clf_dtc, clf_xgb, clf_lr, clf_knn, clf_gnb]
#clf_list = [clf_rdf]
#clf_list = [clf_dtc]
#clf_list = [clf_xgb]
clf_list = [clf_lr]
#clf_list = [clf_knn]
#clf_list = [clf_gnb]
feature_reduction = [17,16,15,14,13,12,11,10,9,8,7,6,5,4,3,2,1]
#data = original_data.copy()
i = 0

# all feature
while i <= 9:
  for count, clf in enumerate(clf_list):

      for num in feature_reduction:

          # restore original data
          data = original_data.copy()

          print()
          print("Number of features:", num)


          # drop target variable
          targetvariable = "HeartDisease"
          data_temp = data.copy()
          #data_temp.drop(columns=targetvariable)
          data_temp = data_temp.drop(labels = ["HeartDisease"], axis = 1)
          #print(data_temp)
          # reduce feature number by num (random)
          df_reducedfeature = data_temp.sample(n=num, axis=1)
          # add target variable again
          df_reducedfeature = pd.concat([data[targetvariable], df_reducedfeature], axis=1)


          print(df_reducedfeature.columns)

          # variation
          row_main_variation(df_reducedfeature, clf_names[count], clf, num, i)
  i += 1

# csv merge
df1 = pd.read_csv('emissions.csv')
df2 = pd.read_csv('emissions_data.csv')
df_final = df1.merge(df2, on='project_name', how='outer')
df_final.to_csv('emissions_full.csv', index=False)



Number of features: 17
Index(['HeartDisease', 'Smoking', 'SleepTime', 'AgeCategory', 'Asthma',
       'PhysicalActivity', 'PhysicalHealth', 'MentalHealth', 'Race',
       'Diabetic', 'DiffWalking', 'BMI', 'GenHealth', 'Stroke',
       'KidneyDisease', 'Sex', 'AlcoholDrinking', 'SkinCancer'],
      dtype='object')

Dataset reduced by [%]: 0
Number of rows: 319795


[codecarbon INFO @ 19:13:48] [setup] RAM Tracking...
[codecarbon INFO @ 19:13:48] [setup] GPU Tracking...
[codecarbon INFO @ 19:13:48] No GPU found.
[codecarbon INFO @ 19:13:48] [setup] CPU Tracking...
[codecarbon WARNING @ 19:13:48] No CPU tracking mode found. Falling back on CPU constant mode.


LogisticRegression


[codecarbon WARNING @ 19:13:50] We saw that you have a Intel(R) Xeon(R) CPU @ 2.20GHz but we don't know it. Please contact us.
[codecarbon INFO @ 19:13:50] CPU Model on constant consumption mode: Intel(R) Xeon(R) CPU @ 2.20GHz
[codecarbon INFO @ 19:13:50] >>> Tracker's metadata:
[codecarbon INFO @ 19:13:50]   Platform system: Linux-5.15.120+-x86_64-with-glibc2.35
[codecarbon INFO @ 19:13:50]   Python version: 3.10.12
[codecarbon INFO @ 19:13:50]   CodeCarbon version: 2.3.1
[codecarbon INFO @ 19:13:50]   Available RAM : 12.678 GB
[codecarbon INFO @ 19:13:50]   CPU count: 2
[codecarbon INFO @ 19:13:50]   CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz
[codecarbon INFO @ 19:13:50]   GPU count: None
[codecarbon INFO @ 19:13:50]   GPU model: None
[codecarbon INFO @ 19:13:50] Energy consumed for RAM : 0.000001 kWh. RAM Power : 4.7543792724609375 W
[codecarbon INFO @ 19:13:50] Energy consumed for all CPUs : 0.000005 kWh. Total CPU Power : 42.5 W
[codecarbon INFO @ 19:13:50] 0.000006 kWh of electric


Dataset reduced by [%]: 20
Number of rows: 255836
LogisticRegression


[codecarbon WARNING @ 19:13:53] We saw that you have a Intel(R) Xeon(R) CPU @ 2.20GHz but we don't know it. Please contact us.
[codecarbon INFO @ 19:13:53] CPU Model on constant consumption mode: Intel(R) Xeon(R) CPU @ 2.20GHz
[codecarbon INFO @ 19:13:53] >>> Tracker's metadata:
[codecarbon INFO @ 19:13:53]   Platform system: Linux-5.15.120+-x86_64-with-glibc2.35
[codecarbon INFO @ 19:13:53]   Python version: 3.10.12
[codecarbon INFO @ 19:13:53]   CodeCarbon version: 2.3.1
[codecarbon INFO @ 19:13:53]   Available RAM : 12.678 GB
[codecarbon INFO @ 19:13:53]   CPU count: 2
[codecarbon INFO @ 19:13:53]   CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz
[codecarbon INFO @ 19:13:53]   GPU count: None
[codecarbon INFO @ 19:13:53]   GPU model: None
[codecarbon INFO @ 19:13:54] Energy consumed for RAM : 0.000001 kWh. RAM Power : 4.7543792724609375 W
[codecarbon INFO @ 19:13:54] Energy consumed for all CPUs : 0.000007 kWh. Total CPU Power : 42.5 W
[codecarbon INFO @ 19:13:54] 0.000008 kWh of electric


Dataset reduced by [%]: 40
Number of rows: 191877
LogisticRegression


[codecarbon WARNING @ 19:13:56] We saw that you have a Intel(R) Xeon(R) CPU @ 2.20GHz but we don't know it. Please contact us.
[codecarbon INFO @ 19:13:56] CPU Model on constant consumption mode: Intel(R) Xeon(R) CPU @ 2.20GHz
[codecarbon INFO @ 19:13:56] >>> Tracker's metadata:
[codecarbon INFO @ 19:13:56]   Platform system: Linux-5.15.120+-x86_64-with-glibc2.35
[codecarbon INFO @ 19:13:56]   Python version: 3.10.12
[codecarbon INFO @ 19:13:56]   CodeCarbon version: 2.3.1
[codecarbon INFO @ 19:13:56]   Available RAM : 12.678 GB
[codecarbon INFO @ 19:13:56]   CPU count: 2
[codecarbon INFO @ 19:13:56]   CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz
[codecarbon INFO @ 19:13:56]   GPU count: None
[codecarbon INFO @ 19:13:56]   GPU model: None
[codecarbon INFO @ 19:13:57] Energy consumed for RAM : 0.000000 kWh. RAM Power : 4.7543792724609375 W
[codecarbon INFO @ 19:13:57] Energy consumed for all CPUs : 0.000003 kWh. Total CPU Power : 42.5 W
[codecarbon INFO @ 19:13:57] 0.000003 kWh of electric


Dataset reduced by [%]: 60
Number of rows: 127918
LogisticRegression


KeyboardInterrupt: ignored